# Lab 1: Marketing Content Generation with LLMs

Welcome to your first hands-on experience with Large Language Models (LLMs)!

**What you'll learn in this lab:**
- How to configure Gemini using the Google Gen AI SDK
- How to write effective prompts for content generation
- How to use system instructions to guide LLM behavior
- How to get structured (JSON) outputs from LLMs
- How to use few-shot prompting to improve results

**Prerequisites:**
- Basic Python knowledge
- A configured Google backend from the README (Vertex AI or Gemini Developer API)

---


##  Understanding Key Concepts

Before we dive in, let's understand some fundamental concepts:

### What is an LLM (Large Language Model)?
An LLM is an AI model trained on vast amounts of text data. It can understand and generate human-like text. Think of it as a very sophisticated autocomplete that can:
- Answer questions
- Write content (emails, articles, code)
- Summarize information
- Transform text from one format to another

### What is a Prompt?
A **prompt** is the input text you send to an LLM. The quality of your prompt directly affects the quality of the response. Good prompts are:
- **Clear**: State exactly what you want
- **Specific**: Include relevant details and context
- **Structured**: Organize information logically

### What are System Instructions?
**System instructions** are special instructions that define the LLM's persona, behavior, and constraints. They're like giving the AI a "job description" before it starts working.

---


## Step 1: Environment setup

Install the pinned requirements using the README. We use the Google Gen AI SDK (`google-genai`), shared by Vertex AI and the Gemini Developer API. Keep credentials in environment variables.


In [ ]:
# Install requirements.txt from the README before opening this notebook.
from pathlib import Path
import sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "lab_support.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside the cloned AI_Trainings repository.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT.name)


In [ ]:
import os
import json
from google.genai import types
from lab_support import get_client, DEFAULT_MODEL


## Step 2: Choose your backend

The README provides separate Vertex AI/ADC and Gemini API key instructions. Vertex AI is the default and requires an explicitly selected project. Model requests may incur charges.


In [ ]:
BACKEND = os.getenv("LAB_BACKEND", "vertex")
MODEL = DEFAULT_MODEL
print("Backend:", BACKEND, "Model:", MODEL)


## Step 3: Create the client

`get_client()` reads your selected authentication path. It never writes credentials to a file. See `lab_support.py` for the short implementation.


In [ ]:
client = get_client()
print("Client configured. The next generation call contacts Google.")


##  Step 4: Create System Instructions

### Add system instructions to optimize the LLM for a marketing content generation assistant

System instructions set the "personality" and "expertise" of the AI. For a marketing assistant, we want to:
- Define its role (marketing expert)
- Set the tone (professional but engaging)
- Establish constraints (brand-appropriate, accurate)

**Example system instruction components:**
- **Role**: "You are a professional marketing copywriter..."
- **Expertise**: "...specializing in retail and consumer products"
- **Tone**: "Use an engaging, friendly tone that resonates with consumers"
- **Constraints**: "Keep content concise and highlight key benefits"


In [ ]:
# System instructions define the AI's role and behavior
system_instructions = """
You are a professional marketing copywriter specializing in retail and consumer products.

Your expertise includes:
- Creating compelling product descriptions
- Writing engaging seasonal and holiday-themed content
- Transforming product features into customer benefits

Guidelines:
- Use an engaging, friendly tone that resonates with everyday consumers
- Highlight key benefits and unique selling points
- Keep content concise but impactful
- Make seasonal content feel festive and relevant
"""

print("System instructions set! The AI will now act as a marketing copywriter.")


## Step 5: Generate with a system instruction

The helper below makes the SDK request explicit and applies the same system instruction to plain text and structured outputs. A JSON schema constrains format; you must still check the facts.


In [ ]:
def generate(prompt, generation_config=None):
    config = generation_config or types.GenerateContentConfig(max_output_tokens=2048)
    config.system_instruction = system_instructions
    response = client.models.generate_content(model=MODEL, contents=prompt, config=config)
    if not response.text:
        raise ValueError("No text returned; inspect the response and model configuration.")
    return response


## Step 6: Basic Prompting - Holiday Content Generation

### Write a prompt that tells the LLM to adjust these feature bullets for an advertisement for this soft drink for the Fourth of July holiday

Now let's put our AI to work! We have product feature bullets and want to transform them into Fourth of July themed marketing content.

**Prompting Best Practices:**
1. **Be specific**: Tell the AI exactly what format/style you want
2. **Provide context**: Include all relevant information (the product features)
3. **Set expectations**: Specify the output format if needed

Below, we'll provide the product features and ask the AI to adapt them for the holiday.


In [ ]:
# Original product feature bullets for Canada Dry Ginger Ale Zero Sugar
feature_bullets = """
    ZERO SUGAR: The great taste of Canada Dry Ginger Ale Zero Sugar is also caffeine-free so you can enjoy it guilt-free any time of day
    RELAXING & REFRESHING: Sip into your comfort zone with Canada Dry Zero Sugar
    CARBONATED SODA: Carbonated soda that tickles your senses with bubbly flavor and refreshing ginger taste that satisfies your thirst every time
    CAFFEINE FREE: The great taste of Canada Dry Ginger Ale without caffeine so you can enjoy it any time of day
    COCKTAIL MIXER: Canada Dry Ginger Ale Zero Sugar is the perfect mixer for delicious, modern cocktails or to enjoy all by itself
"""

print("---Base Prompt---")

# Craft a clear, specific prompt
prompt = f"""
Transform the following product feature bullets into engaging Fourth of July themed 
marketing content. Make it feel festive, patriotic, and perfect for summer celebrations.

Keep the same number of bullet points, but rewrite each one to tie into Independence Day 
themes like BBQs, fireworks, family gatherings, and summer fun.

Original Feature Bullets:
{feature_bullets}

Rewritten Fourth of July Feature Bullets:
"""

# Generate content using the model
response = generate(prompt)
print(response.text)


### What Just Happened?

1. We sent our prompt (with the product features) to the Gemini model
2. The model used its training + our system instructions to generate relevant content
3. The response came back as plain text

Notice how the AI:
- Kept the core product benefits
- Added Fourth of July themes (patriotism, BBQs, fireworks)
- Maintained an engaging, marketing-friendly tone

---


## Step 7: Structured Output (JSON)

### Convert this response to a structured output where each feature bullet is an item in a list

Plain text is great for humans, but what if we need to use this data in an application? That's where **structured output** comes in!

**Why use structured output?**
-  Easy to parse in code (no regex needed!)
-  A defined output structure; still validate parsing and required fields
-  Can be stored in databases directly
-  Works great with APIs and frontend applications

**How it works:**
We define a **schema** (structure) that tells the AI exactly what format to return.

See [Google's documentation](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/control-generated-output) for more on structured output.


In [ ]:
print("---Structured Output---")

# Define the schema for our response
# We want a list of feature bullets, each with a title and description
response_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "title": {
                "type": "string",
                "description": "Short title for the feature (e.g., 'ZERO SUGAR')"
            },
            "description": {
                "type": "string",
                "description": "The Fourth of July themed marketing description"
            }
        },
        "required": ["title", "description"]
    }
}

# Generate with structured output configuration
response = generate(
    prompt, 
    generation_config=types.GenerateContentConfig(
        response_mime_type="application/json",  # Tell the model to return JSON
        response_schema=response_schema          # Provide the structure to follow
    )
)

print(response.text)

parsed = json.loads(response.text)
assert isinstance(parsed, list) and parsed
assert all(isinstance(item.get("title"), str) and isinstance(item.get("description"), str) for item in parsed)
print("JSON shape validated. Now check each claim against the input features.")


###  Understanding the Schema

The schema we defined tells the AI:
- Return an **array** (list) of items
- Each item is an **object** with two properties:
  - `title`: A string for the feature name
  - `description`: A string for the marketing copy

Now you can easily parse this JSON in Python:
```python
import json
features = json.loads(response.text)
for feature in features:
    print(f"{feature['title']}: {feature['description']}")
```

---


##  Step 8: Few-Shot Prompting

### Use another soft drink's description to create an example to improve the response

**What is Few-Shot Prompting?**

Few-shot prompting is a technique where you provide examples of the desired input-output pairs before asking for a new generation. It's like showing someone "here's what I want" before asking them to do it.

**Types of prompting:**
- **Zero-shot**: No examples, just instructions (what we did before)
- **One-shot**: One example provided
- **Few-shot**: Multiple examples provided (typically 2-5)

**Why does it work?**
Examples help the AI understand:
- The exact format you want
- The tone and style to use
- The level of detail expected
- How to transform the input


In [ ]:
# Example input: Sprite feature bullets (original)
soft_drink_2 = """
    Quench your thirst with refreshing Sprite\u00a0soda\u200b
    Clear, crisp\u00a0lemon-lime soda\u00a0will keep you invigorated and inspired\u200b
    A delicious\u00a0citrus taste\u00a0that knows how to keep things cool\u200b
    Caffeine-free, full of 100% natural flavors
    12 fl oz can 12 pack to help you cut through the noise
"""

# Example output: How we want the Fourth of July version to look
fourth_of_july_sd2 = """
    Refreshing Sprite soda will keep your Independence Day celebrations light and bright.
    Its clear, crisp lemon-lime burst adds the perfect sparkle to your backyard BBQ or picnic.
    A delicious citrus taste that adds a cool blast - ideal for hot summer days and fiery fireworks.
    Caffeine-free and made with 100% natural flavors, the whole family can enjoy it all day long.
    Stock up for your festivities with the convenient 12-pack of 12 fl oz cans - plenty of crisp refreshment for every guest!
"""

print("---Few Shot Prompting---")

# Build a few-shot prompt with an example
prompt = f"""
Transform product feature bullets into Fourth of July themed marketing content.

Here is an example of how to do this:

EXAMPLE INPUT (Original Sprite Features):
{soft_drink_2}

EXAMPLE OUTPUT (Fourth of July Themed):
{fourth_of_july_sd2}

Now, apply the same transformation to this product:

INPUT (Original Canada Dry Features):
{feature_bullets}

OUTPUT (Fourth of July Themed):
"""

response = generate(prompt)
print(response.text)


### Comparing Results

Compare the few-shot result to the zero-shot result from Step 6. You might notice:
- More consistent formatting; Note, all the bullet titles now remain the same but the content is still modified based on our theme prompt
- Similar tone and style to the example
- Better alignment with your expectations

**When to use few-shot prompting:**
- When you need very specific formatting
- When zero-shot results aren't quite right
- When you want consistent output across multiple requests
- When the task is complex or nuanced

---


# 🧪 LAB WORK

Now it's your turn! Complete the following exercises to practice what you've learned.

---

### Exercise 1: Create a Marketing Email
**Task:** Using Vertex AI, convert the feature bullets for Canada Dry Ginger Ale into a marketing email.

**Hints:**
- Include a catchy subject line
- Add a greeting and sign-off
- Make it feel personal and engaging
- Include a call-to-action (e.g., "Shop now!")


In [ ]:
print("---Lab Solution 1---")

# Your solution here!
# Hint: Create a prompt that asks for an email format

email_prompt = f"""
Create a marketing email promoting Canada Dry Ginger Ale Zero Sugar.

Use these product features as your source material:
{feature_bullets}

The email should include:
- A catchy subject line
- A friendly greeting
- 2-3 paragraphs highlighting the key benefits
- A clear call-to-action
- A professional sign-off

Make it feel personal and engaging, targeting health-conscious consumers who love refreshing drinks.
"""

response = generate(email_prompt)
print(response.text)


---

### Exercise 2: Out of Office Message
**Task:** Write a one sentence out of office message for Microsoft Teams for the Fourth of July with a polite tone.

**Hints:**
- Keep it brief (one sentence)
- Mention the holiday
- Include when you'll be back
- Maintain a professional but friendly tone


In [ ]:
print("---Lab Solution 2---")

# Your solution here!

ooo_prompt = """
Write a one sentence out of office message for Microsoft Teams for the Fourth of July holiday.

Requirements:
- Exactly one sentence
- Polite and professional tone
- Mention the holiday
- Indicate returning on July 5th
"""

response = generate(ooo_prompt)
print(response.text)


---

### Exercise 3: Structured Output - BBQ Pairings
**Task:** Generate a list of common BBQ soft drink and meal combinations, in the format `{drink: "Coca Cola", meal: "Hot Dogs"}`

**Hints:**
- Use structured output (JSON)
- Define a schema for the drink-meal pairs
- Ask for 5-10 combinations


In [ ]:
print("---Lab Solution 3---")

# Your solution here!

bbq_prompt = """
Generate a list of 8 classic BBQ soft drink and meal pairings that would be popular at 
an American summer cookout. Include a variety of drinks and meals.
"""

# Define the schema for drink-meal pairs
bbq_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "drink": {
                "type": "string",
                "description": "Name of the soft drink"
            },
            "meal": {
                "type": "string",
                "description": "Name of the BBQ food item"
            }
        },
        "required": ["drink", "meal"]
    }
}

response = generate(
    bbq_prompt,
    generation_config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=bbq_schema
    )
)

print(response.text)

# Bonus: Pretty print the results
import json
pairings = json.loads(response.text)
print("\n📋 BBQ Pairings:")
for i, pair in enumerate(pairings, 1):
    print(f"  {i}. {pair['drink']} + {pair['meal']}")


---

# 🎉 Congratulations!

You've completed Lab 1! Here's what you learned:

| Concept | What You Learned |
|---------|------------------|
| **Vertex AI Setup** | How to initialize and configure the SDK |
| **System Instructions** | How to define the AI's role and behavior |
| **Basic Prompting** | How to write clear, effective prompts |
| **Structured Output** | How to get JSON responses with schemas |
| **Few-Shot Prompting** | How to use examples to improve output quality |

##  What's Next?

In the next session, we'll learn how to use **vector databases** to enable powerful semantic search capabilities for our AI applications.

##  Additional Resources

- [Vertex AI Documentation](https://cloud.google.com/vertex-ai/docs)
- [Gemini API Reference](https://cloud.google.com/vertex-ai/generative-ai/docs/model-reference/gemini)
- [Prompt Engineering Guide](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/prompts/introduction-prompt-design)
- [Structured Output Guide](https://cloud.google.com/vertex-ai/generative-ai/docs/multimodal/control-generated-output)
